# Positional Encodings — Theory Notebook

Interactive implementations covering:
1. Sinusoidal PE — formula, frequency structure, linear offset property
2. Dot-product decay — PE(pos)·PE(pos') depends only on offset
3. Learned absolute PE — undertrained tail simulation
4. RoPE — 2D rotation, d-dimensional block-diagonal, efficient implementation
5. RoPE frequency analysis — wavelengths, dead dimensions
6. ALiBi — slopes, bias matrix, local vs global heads
7. T5 relative bias — bucket formula
8. RoPE scaling — Position Interpolation, NTK-aware, YaRN
9. Extrapolation comparison — sinusoidal vs RoPE vs ALiBi
10. PE comparison dashboard

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_matrix(name, M, decimals=4):
    print(f"\n{name} ({M.shape[0]}×{M.shape[1]}):")
    for row in M:
        print("  [" + "  ".join(f"{v:>{decimals+4}.{decimals}f}" for v in row) + "]")

def print_vector(name, v, decimals=4):
    print(f"{name}: [{', '.join(f'{x:.{decimals}f}' for x in v)}]")

## 1. Sinusoidal Positional Encoding

$$\text{PE}(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad \text{PE}(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

Each dimension pair (2i, 2i+1) oscillates at a different frequency, from wavelength 2π to 2π·10000.

In [ ]:
# === 1a. Sinusoidal PE — Core Implementation ===

def sinusoidal_pe(n, d, base=10000):
    """
    Generate sinusoidal positional encoding.
    
    PE(pos, 2i)   = sin(pos / base^(2i/d))
    PE(pos, 2i+1) = cos(pos / base^(2i/d))
    
    Returns: (n, d) matrix
    """
    pos = np.arange(n)[:, None]           # (n, 1)
    i = np.arange(0, d, 2)[None, :]       # (1, d/2)
    angles = pos / base ** (i / d)        # (n, d/2)
    
    pe = np.zeros((n, d))
    pe[:, 0::2] = np.sin(angles)          # even dims: sin
    pe[:, 1::2] = np.cos(angles)          # odd dims: cos
    return pe

# Generate for small example
n, d = 8, 16
pe = sinusoidal_pe(n, d)

print(f"=== Sinusoidal PE (n={n}, d={d}) ===")
print(f"Shape: {pe.shape}")
print(f"\nFirst 4 positions, first 8 dimensions:")
print_matrix("PE[0:4, 0:8]", pe[:4, :8])

# Verify PE values at specific positions
pos, dim_pair = 3, 2
omega = 1.0 / (10000 ** (2 * dim_pair / d))
expected_sin = np.sin(pos * omega)
expected_cos = np.cos(pos * omega)
print(f"\nVerification: pos={pos}, dim_pair i={dim_pair}")
print(f"  ω_i = 1/10000^({2*dim_pair}/{d}) = {omega:.6f}")
print(f"  PE({pos}, {2*dim_pair}) = sin({pos} · {omega:.6f}) = {expected_sin:.6f}  (got {pe[pos, 2*dim_pair]:.6f}) ✓")
print(f"  PE({pos}, {2*dim_pair+1}) = cos({pos} · {omega:.6f}) = {expected_cos:.6f}  (got {pe[pos, 2*dim_pair+1]:.6f}) ✓")

In [ ]:
# === 1b. Frequency Structure ===

d = 512
base = 10000
dim_pairs = np.arange(d // 2)
frequencies = 1.0 / (base ** (2 * dim_pairs / d))
wavelengths = 2 * np.pi / frequencies

print(f"=== Frequency Structure (d={d}, base={base}) ===")
print(f"{'Pair i':>8} {'Frequency ωᵢ':>16} {'Wavelength λᵢ':>16} {'Interpretation':>20}")
print("-" * 65)
for i in [0, 1, 5, 10, 32, 64, 128, 200, 255]:
    interp = "local" if wavelengths[i] < 100 else ("medium" if wavelengths[i] < 10000 else "global")
    print(f"{i:>8} {frequencies[i]:>16.8f} {wavelengths[i]:>16.1f} {interp:>20}")

print(f"\nShortest wavelength (pair 0): {wavelengths[0]:.2f} positions")
print(f"Longest wavelength (pair {d//2-1}): {wavelengths[-1]:.0f} positions")
print(f"Range: {wavelengths[-1]/wavelengths[0]:.0f}×")

In [ ]:
# === 1c. Sinusoidal PE Visualisation ===

if HAS_MPL:
    n_vis, d_vis = 128, 64
    pe_vis = sinusoidal_pe(n_vis, d_vis)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # (a) Full PE heatmap
    im = axes[0].imshow(pe_vis, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0].set_xlabel('Dimension')
    axes[0].set_ylabel('Position')
    axes[0].set_title(f'Sinusoidal PE ({n_vis}×{d_vis})')
    plt.colorbar(im, ax=axes[0], shrink=0.8)
    
    # (b) Individual dimension traces
    for pair_i in [0, 4, 8, 16, 31]:
        axes[1].plot(pe_vis[:, 2*pair_i], label=f'dim {2*pair_i} (i={pair_i})', alpha=0.7)
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('PE value')
    axes[1].set_title('PE values across positions (sin dims)')
    axes[1].legend(fontsize=7)
    axes[1].grid(True, alpha=0.3)
    
    # (c) Wavelength vs dimension pair
    d_full = 512
    pairs = np.arange(d_full // 2)
    wl = 2 * np.pi / (1.0 / (10000 ** (2 * pairs / d_full)))
    axes[2].semilogy(pairs, wl)
    axes[2].axhline(y=2048, color='r', ls='--', alpha=0.5, label='n=2048')
    axes[2].axhline(y=4096, color='orange', ls='--', alpha=0.5, label='n=4096')
    axes[2].set_xlabel('Dimension pair i')
    axes[2].set_ylabel('Wavelength (positions)')
    axes[2].set_title(f'Wavelengths (d={d_full})')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available; skipping visualisation")

In [ ]:
# === 1d. Linear Relationship: PE(pos+k) = M_k · PE(pos) ===

d = 8
base = 10000
pe = sinusoidal_pe(10, d, base)

print("=== Linear Offset Property ===")
print("PE(pos+k) = M_k · PE(pos) where M_k is block-diagonal rotation\n")

# Build M_k for offset k=1
k = 1
M_k = np.zeros((d, d))
for pair_i in range(d // 2):
    omega = 1.0 / (base ** (2 * pair_i / d))
    angle = k * omega
    c, s = np.cos(angle), np.sin(angle)
    idx = 2 * pair_i
    # For (sin, cos) convention: rotation matrix that maps PE(pos) → PE(pos+k)
    M_k[idx, idx] = c
    M_k[idx, idx+1] = s
    M_k[idx+1, idx] = -s
    M_k[idx+1, idx+1] = c

print_matrix("M_1 (offset=1 rotation matrix)", M_k)

# Verify: M_1 · PE(pos) ≈ PE(pos+1)
print(f"\nVerification for multiple positions:")
print(f"{'pos':>5} {'||M_1·PE(pos) - PE(pos+1)||':>30}")
print("-" * 40)
for pos in range(8):
    predicted = M_k @ pe[pos]
    actual = pe[pos + 1]
    error = np.linalg.norm(predicted - actual)
    print(f"{pos:>5} {error:>30.2e}")

# Verify with larger offset
k = 3
M_3 = np.zeros((d, d))
for pair_i in range(d // 2):
    omega = 1.0 / (base ** (2 * pair_i / d))
    angle = k * omega
    c, s = np.cos(angle), np.sin(angle)
    idx = 2 * pair_i
    M_3[idx, idx] = c
    M_3[idx, idx+1] = s
    M_3[idx+1, idx] = -s
    M_3[idx+1, idx+1] = c

print(f"\nVerification for offset k=3:")
for pos in range(6):
    error = np.linalg.norm(M_3 @ pe[pos] - pe[pos + 3])
    print(f"  M_3 · PE({pos}) ≈ PE({pos+3}): error = {error:.2e}")

# Also verify M_3 = M_1^3
M_1_cubed = np.linalg.matrix_power(M_k, 3)
print(f"\n||M_3 - M_1³|| = {np.linalg.norm(M_3 - M_1_cubed):.2e}  (composition property ✓)")

## 2. Dot-Product Decay

$$\text{PE}(pos) \cdot \text{PE}(pos') = \sum_{i=0}^{d/2-1} \cos\!\left((pos - pos') \cdot \omega_i\right)$$

The dot product depends **only** on the relative offset — sinusoidal PE implicitly encodes relative distance.

In [ ]:
# === 2. Dot-Product Decay Analysis ===

d = 512
base = 10000
n = 200
pe = sinusoidal_pe(n, d, base)

# Compute dot products PE(0)·PE(k) for various offsets
offsets = np.arange(n)
dots_from_0 = pe[0] @ pe.T  # PE(0)·PE(k) for all k

# Verify: dot product depends only on offset (not absolute position)
print("=== Dot Product Depends Only on Offset ===")
print(f"{'offset':>8} {'PE(0)·PE(k)':>14} {'PE(5)·PE(5+k)':>16} {'PE(50)·PE(50+k)':>18} {'Match':>8}")
print("-" * 70)
for k in [0, 1, 2, 5, 10, 20, 50]:
    d1 = pe[0] @ pe[k]
    d2 = pe[5] @ pe[5 + k] if 5 + k < n else float('nan')
    d3 = pe[50] @ pe[50 + k] if 50 + k < n else float('nan')
    match = np.isclose(d1, d2, atol=1e-8) and np.isclose(d1, d3, atol=1e-8)
    print(f"{k:>8} {d1:>14.4f} {d2:>16.4f} {d3:>18.4f} {'✓' if match else '✗':>8}")

# Analytical formula: sum of cos((pos-pos')·ω_i)
print(f"\n=== Analytical vs Computed ===")
for k in [0, 1, 5, 50, 100]:
    computed = pe[0] @ pe[k]
    # Analytical: sum_i cos(k * omega_i)
    dim_pairs = np.arange(d // 2)
    omegas = 1.0 / (base ** (2 * dim_pairs / d))
    analytical = np.sum(np.cos(k * omegas))
    error = abs(computed - analytical)
    print(f"  k={k:>3}: computed={computed:>10.4f}  analytical={analytical:>10.4f}  error={error:.2e}")

In [ ]:
# === 2b. Dot-Product Decay Visualisation ===

if HAS_MPL:
    d = 512
    n = 500
    pe = sinusoidal_pe(n, d)
    
    offsets = np.arange(n)
    dots = np.array([pe[0] @ pe[k] for k in offsets])
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # (a) Full decay curve
    axes[0].plot(offsets, dots, 'b-', alpha=0.7, linewidth=0.5)
    axes[0].axhline(y=0, color='gray', ls='--', alpha=0.3)
    axes[0].set_xlabel('Offset k')
    axes[0].set_ylabel('PE(0) · PE(k)')
    axes[0].set_title(f'Dot-product decay (d={d})')
    axes[0].set_xlim(0, n)
    axes[0].grid(True, alpha=0.3)
    
    # (b) Zoom into first 50 positions
    axes[1].plot(offsets[:50], dots[:50], 'b-o', markersize=3, alpha=0.7)
    axes[1].axhline(y=0, color='gray', ls='--', alpha=0.3)
    axes[1].set_xlabel('Offset k')
    axes[1].set_ylabel('PE(0) · PE(k)')
    axes[1].set_title('Zoom: first 50 positions')
    axes[1].grid(True, alpha=0.3)
    
    # (c) Dot product matrix (similarity between all position pairs)
    n_small = 64
    pe_small = sinusoidal_pe(n_small, d)
    dot_matrix = pe_small @ pe_small.T
    im = axes[2].imshow(dot_matrix, cmap='RdBu_r', aspect='auto')
    axes[2].set_xlabel('Position')
    axes[2].set_ylabel('Position')
    axes[2].set_title(f'PE dot-product matrix ({n_small}×{n_small})')
    plt.colorbar(im, ax=axes[2], shrink=0.8)
    
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available")

## 3. RoPE — Rotary Positional Encoding

Rotation matrix for position m, dimension pair i:

$$R_m^{(i)} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}, \quad \theta_i = \frac{1}{\text{base}^{2i/d}}$$

Key property: $(R_m q)^\top (R_n k) = q^\top R_{n-m} k$ — dot product depends only on relative offset.

In [ ]:
# === 3a. RoPE — 2D Case (Building Intuition) ===

print("=== RoPE 2D Case ===")
print("d=2, single frequency θ\n")

theta = 1.0  # radians per position step

def rotation_2d(angle):
    """2D rotation matrix."""
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])

# Query and key vectors
q = np.array([1.0, 0.5])
k = np.array([0.3, 0.8])

print(f"Original vectors: q={q}, k={k}")
print(f"Original dot product: q·k = {q @ k:.6f}\n")

# Rotate at different positions
print(f"{'pos_q':>6} {'pos_k':>6} {'offset':>8} {'dot(R_m·q, R_n·k)':>20}")
print("-" * 45)
dots_by_offset = {}
for m in range(5):
    for n in range(5):
        Rm = rotation_2d(m * theta)
        Rn = rotation_2d(n * theta)
        q_rot = Rm @ q
        k_rot = Rn @ k
        dot_val = q_rot @ k_rot
        offset = m - n
        dots_by_offset.setdefault(offset, []).append(dot_val)
        if m < 3 and n < 3:
            print(f"{m:>6} {n:>6} {offset:>8} {dot_val:>20.8f}")

# Verify: same offset → same dot product
print(f"\n=== Verification: dot depends only on offset ===")
for offset in sorted(dots_by_offset.keys()):
    vals = dots_by_offset[offset]
    all_equal = np.allclose(vals, vals[0], atol=1e-12)
    print(f"  offset {offset:>3}: {vals[0]:>12.8f}  (all {len(vals)} values equal: {all_equal} ✓)")

# Mathematical proof visualization
print(f"\n=== Proof: (R_m q)^T (R_n k) = q^T R_(n-m) k ===")
m, n = 3, 1
Rm = rotation_2d(m * theta)
Rn = rotation_2d(n * theta)
R_nm = rotation_2d((n - m) * theta)
lhs = (Rm @ q) @ (Rn @ k)
rhs = q @ (R_nm @ k)
print(f"  LHS: (R_{m} q)^T (R_{n} k) = {lhs:.10f}")
print(f"  RHS: q^T R_{n-m} k          = {rhs:.10f}")
print(f"  Equal: {np.isclose(lhs, rhs)} ✓")

In [ ]:
# === 3b. RoPE — Full d-Dimensional Implementation ===

def compute_rope_freqs(d, base=10000):
    """Compute RoPE frequencies θ_i = 1/base^(2i/d)."""
    i = np.arange(0, d, 2)  # dimension pair indices
    return 1.0 / (base ** (i / d))  # (d/2,)

def build_rope_matrix(pos, thetas):
    """Build full d×d block-diagonal rotation matrix R_pos."""
    d = len(thetas) * 2
    R = np.eye(d)
    for i, theta in enumerate(thetas):
        angle = pos * theta
        c, s = np.cos(angle), np.sin(angle)
        idx = 2 * i
        R[idx, idx] = c
        R[idx, idx+1] = -s
        R[idx+1, idx] = s
        R[idx+1, idx+1] = c
    return R

def apply_rope_efficient(q, positions, base=10000):
    """
    Apply RoPE to vectors efficiently (no matrix construction).
    q: (n, d), positions: (n,)
    """
    n, d = q.shape
    i = np.arange(0, d, 2) / d       # (d/2,) — normalized pair indices
    freqs = 1.0 / (base ** i)        # (d/2,) — frequencies
    angles = positions[:, None] * freqs[None, :]  # (n, d/2)
    
    cos_a = np.cos(angles)  # (n, d/2)
    sin_a = np.sin(angles)  # (n, d/2)
    
    q_even = q[:, 0::2]  # (n, d/2)
    q_odd  = q[:, 1::2]  # (n, d/2)
    
    q_rot = np.zeros_like(q)
    q_rot[:, 0::2] = q_even * cos_a - q_odd * sin_a
    q_rot[:, 1::2] = q_even * sin_a + q_odd * cos_a
    return q_rot

# Test with d=8
d = 8
base = 10000
thetas = compute_rope_freqs(d, base)
print(f"=== RoPE d={d}, base={base} ===")
print(f"Frequencies θ_i: {thetas.round(6)}")
print(f"Wavelengths λ_i: {(2*np.pi/thetas).round(1)}")

# Build explicit rotation matrix for position 3
pos = 3
R_3 = build_rope_matrix(pos, thetas)
print_matrix(f"R_{pos} (block-diagonal rotation)", R_3)

# Verify it's orthogonal
print(f"\nR_3 is orthogonal: ||R^T R - I|| = {np.linalg.norm(R_3.T @ R_3 - np.eye(d)):.2e}")
print(f"det(R_3) = {np.linalg.det(R_3):.6f} (should be 1.0)")

# Compare explicit vs efficient implementation
np.random.seed(42)
n_test = 6
Q_test = np.random.randn(n_test, d)
positions = np.arange(n_test, dtype=float)

# Explicit: multiply each row by its rotation matrix
Q_explicit = np.zeros_like(Q_test)
for idx in range(n_test):
    R = build_rope_matrix(positions[idx], thetas)
    Q_explicit[idx] = R @ Q_test[idx]

# Efficient: elementwise cos/sin
Q_efficient = apply_rope_efficient(Q_test, positions, base)

print(f"\n=== Explicit vs Efficient ===")
print(f"Max difference: {np.max(np.abs(Q_explicit - Q_efficient)):.2e}")
print(f"Match: {np.allclose(Q_explicit, Q_efficient)} ✓")

In [ ]:
# === 3c. RoPE — Verify Relative Position Property (d-dimensional) ===

d = 8
base = 10000
np.random.seed(42)
q = np.random.randn(d)
k = np.random.randn(d)
thetas = compute_rope_freqs(d, base)

print(f"=== RoPE Relative Position Property (d={d}) ===")
print(f"Verify: dot(R_m q, R_n k) depends only on m−n\n")

print(f"{'m':>4} {'n':>4} {'m-n':>5} {'dot(R_m q, R_n k)':>20}")
print("-" * 38)

offset_dots = {}
for m in range(8):
    for n in range(8):
        Rm = build_rope_matrix(m, thetas)
        Rn = build_rope_matrix(n, thetas)
        dot_val = (Rm @ q) @ (Rn @ k)
        offset = m - n
        offset_dots.setdefault(offset, []).append(dot_val)
        if m < 4 and n < 4:
            print(f"{m:>4} {n:>4} {offset:>5} {dot_val:>20.10f}")

print(f"\n=== All offsets ===")
for offset in sorted(offset_dots.keys()):
    vals = offset_dots[offset]
    spread = max(vals) - min(vals)
    print(f"  offset {offset:>3}: value = {vals[0]:.10f}  spread = {spread:.2e}  ({'✓' if spread < 1e-10 else '✗'})")

# Norm preservation check
print(f"\n=== Norm Preservation ===")
for m in range(5):
    Rm = build_rope_matrix(m, thetas)
    q_rot = Rm @ q
    print(f"  ||q|| = {np.linalg.norm(q):.6f}, ||R_{m}q|| = {np.linalg.norm(q_rot):.6f}  ✓")

In [ ]:
# === 3d. RoPE Rotation Visualisation ===

if HAS_MPL:
    d = 8
    thetas = compute_rope_freqs(d, base=10000)
    np.random.seed(42)
    q = np.random.randn(d)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # (a) 2D projections at different positions
    positions = np.arange(20)
    for pair_i, ax_idx in [(0, (0,0)), (1, (0,1)), (2, (1,0)), (3, (1,1))]:
        ax = axes[ax_idx]
        # Track how this dimension pair rotates with position
        coords = []
        for pos in positions:
            R = build_rope_matrix(pos, thetas)
            q_rot = R @ q
            coords.append([q_rot[2*pair_i], q_rot[2*pair_i+1]])
        coords = np.array(coords)
        
        # Draw unit circle
        circle = plt.Circle((0, 0), np.sqrt(q[2*pair_i]**2 + q[2*pair_i+1]**2), 
                           fill=False, color='gray', linestyle='--', alpha=0.3)
        ax.add_patch(circle)
        
        # Plot trajectory
        ax.plot(coords[:, 0], coords[:, 1], 'b-', alpha=0.4)
        scatter = ax.scatter(coords[:, 0], coords[:, 1], c=positions, cmap='viridis', s=40, zorder=5)
        ax.scatter(coords[0, 0], coords[0, 1], c='red', s=100, marker='*', zorder=6, label='pos=0')
        
        ax.set_xlabel(f'dim {2*pair_i}')
        ax.set_ylabel(f'dim {2*pair_i+1}')
        wl = 2*np.pi / thetas[pair_i]
        ax.set_title(f'Pair i={pair_i} (θ={thetas[pair_i]:.4f}, λ={wl:.0f})')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    
    plt.suptitle('RoPE: 2D Projections of Rotated Query Vector', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available")

## 4. RoPE Frequency Analysis

For base=10000, d=128: wavelengths span from 2π ≈ 6.3 to 2π·10000 ≈ 62,832.

High-frequency dimensions (low i) complete many rotations within training length → **aliasing** risk.
Low-frequency dimensions (high i) barely rotate → **undertrained** for positional discrimination.

In [ ]:
# === 4. RoPE Frequency Analysis ===

d = 128
base = 10000
n_train = 4096

dim_pairs = np.arange(d // 2)
thetas = 1.0 / (base ** (2 * dim_pairs / d))
wavelengths = 2 * np.pi / thetas

# How many full rotations does each dimension complete within n_train?
rotations_in_train = n_train / wavelengths

print(f"=== RoPE Frequency Analysis (d={d}, base={base}, n_train={n_train}) ===\n")
print(f"{'Pair i':>8} {'θ_i':>12} {'Wavelength':>12} {'Rotations in n_train':>22} {'Zone':>12}")
print("-" * 72)

for i in [0, 4, 8, 16, 24, 32, 40, 48, 56, 63]:
    zone = "HIGH FREQ" if rotations_in_train[i] > 1 else ("MID FREQ" if rotations_in_train[i] > 0.1 else "LOW FREQ")
    print(f"{i:>8} {thetas[i]:>12.6f} {wavelengths[i]:>12.1f} {rotations_in_train[i]:>22.4f} {zone:>12}")

# Count dimensions in each zone
high_freq = np.sum(rotations_in_train > 1)
mid_freq = np.sum((rotations_in_train > 0.1) & (rotations_in_train <= 1))
low_freq = np.sum(rotations_in_train <= 0.1)

print(f"\n=== Zone Summary ===")
print(f"  High frequency (>1 rotation in n_train): {high_freq} pairs ({2*high_freq} dims)")
print(f"  Mid frequency  (0.1–1 rotations):        {mid_freq} pairs ({2*mid_freq} dims)")
print(f"  Low frequency  (<0.1 rotation):           {low_freq} pairs ({2*low_freq} dims)")
print(f"\n  'Dead' dimensions for n_train=4096: {low_freq} pairs barely rotate → poor positional signal")

# Effect of different context lengths
print(f"\n=== Context Length Effect ===")
for n in [512, 2048, 4096, 8192, 32768, 131072]:
    dead = np.sum(n / wavelengths < 0.1)
    useful = d // 2 - dead
    print(f"  n={n:>7}: {useful:>3}/{d//2} pairs useful, {dead:>3} effectively dead")

In [ ]:
# === 4b. RoPE Frequency Visualisation ===

if HAS_MPL:
    d = 128
    base = 10000
    dim_pairs = np.arange(d // 2)
    thetas = 1.0 / (base ** (2 * dim_pairs / d))
    wavelengths = 2 * np.pi / thetas
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # (a) Wavelength vs dimension pair
    axes[0].semilogy(dim_pairs, wavelengths, 'b-', linewidth=2)
    for n_train, color, label in [(2048, 'orange', 'n=2K'), (4096, 'red', 'n=4K'), (32768, 'green', 'n=32K')]:
        axes[0].axhline(y=n_train, color=color, ls='--', alpha=0.5, label=label)
    axes[0].set_xlabel('Dimension pair i')
    axes[0].set_ylabel('Wavelength λ_i (positions)')
    axes[0].set_title('RoPE Wavelengths')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # (b) Rotations completed within training length
    n_train = 4096
    rots = n_train / wavelengths
    colors = ['green' if r > 1 else ('orange' if r > 0.1 else 'red') for r in rots]
    axes[1].bar(dim_pairs, rots, color=colors, alpha=0.7)
    axes[1].axhline(y=1, color='gray', ls='--', alpha=0.5, label='1 full rotation')
    axes[1].axhline(y=0.1, color='gray', ls=':', alpha=0.5, label='0.1 rotation')
    axes[1].set_xlabel('Dimension pair i')
    axes[1].set_ylabel(f'Rotations in n_train={n_train}')
    axes[1].set_title('Positional Signal Strength')
    axes[1].set_yscale('log')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    # (c) dot(R_m q, R_0 k) as function of m, per dimension pair
    d_small = 8
    thetas_s = compute_rope_freqs(d_small, base)
    np.random.seed(42)
    q = np.random.randn(d_small)
    k = np.random.randn(d_small)
    
    positions = np.arange(50)
    for pair_i in range(d_small // 2):
        dots = []
        for m in positions:
            # Only rotate pair_i
            angle_m = m * thetas_s[pair_i]
            c_m, s_m = np.cos(angle_m), np.sin(angle_m)
            q2i = q[2*pair_i] * c_m - q[2*pair_i+1] * s_m
            q2i1 = q[2*pair_i] * s_m + q[2*pair_i+1] * c_m
            dot = q2i * k[2*pair_i] + q2i1 * k[2*pair_i+1]
            dots.append(dot)
        axes[2].plot(positions, dots, label=f'pair {pair_i} (λ={2*np.pi/thetas_s[pair_i]:.0f})', alpha=0.7)
    
    axes[2].set_xlabel('Position m (k at pos 0)')
    axes[2].set_ylabel('Pair contribution to dot product')
    axes[2].set_title(f'Per-pair dot product (d={d_small})')
    axes[2].legend(fontsize=7)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available")

## 5. ALiBi — Attention with Linear Biases

$$S_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}} - m_h \cdot (i - j)$$

Head slopes: $m_h = 1 / 2^{h \cdot 8/H}$ — geometric sequence giving local and global heads.

In [ ]:
# === 5a. ALiBi — Slopes and Bias Matrix ===

def alibi_slopes(num_heads):
    """Compute ALiBi slopes: m_h = 1 / 2^(h * 8/H)."""
    h = np.arange(1, num_heads + 1)
    return 1.0 / (2 ** (h * 8 / num_heads))

def alibi_bias_matrix(n, slopes):
    """Build ALiBi bias matrices for causal attention. Returns (H, n, n)."""
    dist = np.arange(n)[None, :] - np.arange(n)[:, None]  # (n, n): j - i
    dist = np.minimum(dist, 0)  # causal: only past (non-positive values)
    return dist[None, :, :] * slopes[:, None, None]  # (H, n, n)

# Compute for H=8
H = 8
slopes = alibi_slopes(H)
print(f"=== ALiBi Slopes (H={H}) ===")
print(f"{'Head h':>8} {'Slope m_h':>12} {'1/m_h':>10} {'Attention range':>18}")
print("-" * 52)
for h in range(H):
    effective_range = 10 / slopes[h]  # score penalty of -10 at this distance
    label = "LOCAL" if effective_range < 100 else ("MID" if effective_range < 5000 else "GLOBAL")
    print(f"{h+1:>8} {slopes[h]:>12.6f} {1/slopes[h]:>10.1f} {label:>18}")

# Build bias matrix for n=8
n = 8
biases = alibi_bias_matrix(n, slopes)
print(f"\n=== Bias Matrix for Head 1 (slope={slopes[0]:.4f}, most local) ===")
print_matrix("B_1", biases[0], decimals=2)

print(f"\n=== Bias Matrix for Head 8 (slope={slopes[-1]:.6f}, most global) ===")
print_matrix("B_8", biases[-1], decimals=4)

In [ ]:
# === 5b. ALiBi Visualisation ===

if HAS_MPL:
    H = 8
    n = 32
    slopes = alibi_slopes(H)
    biases = alibi_bias_matrix(n, slopes)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    for h in range(H):
        ax = axes[h // 4, h % 4]
        im = ax.imshow(biases[h], aspect='auto', cmap='RdBu_r',
                       vmin=-5, vmax=0)
        ax.set_title(f'Head {h+1} (m={slopes[h]:.4f})')
        ax.set_xlabel('Key pos')
        ax.set_ylabel('Query pos')
        if h % 4 == 3:
            plt.colorbar(im, ax=ax, shrink=0.8)
    
    plt.suptitle(f'ALiBi Bias Matrices (H={H}, n={n})', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Also show attention weight distribution for one query position
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Simulate attention with random content scores + ALiBi
    np.random.seed(42)
    d_k = 64
    Q = np.random.randn(n, d_k) * 0.3
    K = np.random.randn(n, d_k) * 0.3
    content_scores = Q @ K.T / np.sqrt(d_k)  # (n, n)
    
    # Causal mask
    causal = np.triu(np.ones((n, n)) * (-1e9), k=1)
    
    query_pos = n - 1  # last token
    for h_idx, h in enumerate([0, 7]):  # most local vs most global
        scores = content_scores[query_pos] + causal[query_pos] + biases[h, query_pos]
        weights = np.exp(scores - np.max(scores))
        weights = weights / weights.sum()
        
        axes[h_idx].bar(range(n), weights, alpha=0.7)
        axes[h_idx].set_xlabel('Key position')
        axes[h_idx].set_ylabel('Attention weight')
        label = "LOCAL" if h == 0 else "GLOBAL"
        axes[h_idx].set_title(f'Head {h+1} ({label}, m={slopes[h]:.4f})')
        axes[h_idx].grid(True, alpha=0.3)
    
    plt.suptitle(f'ALiBi Attention Weights from Query Position {query_pos}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available")

## 6. T5 Relative Bias — Bucket Formula

T5 uses logarithmic bucketing: exact resolution for nearby tokens (δ < 16), logarithmic for distant tokens.

$$\text{bucket}(\delta) = \begin{cases} \delta & \text{if } \delta < 16 \\ 16 + \lfloor \log(\delta/16) / \log(\text{max\_dist}/16) \times 16 \rfloor & \text{otherwise}\end{cases}$$

In [ ]:
# === 6. T5 Relative Bias Bucketing ===

def t5_bucket(delta, num_buckets=32, max_distance=128):
    """
    T5-style bucking of relative distances.
    First half of buckets: exact (one per offset).
    Second half: logarithmically spaced.
    """
    half = num_buckets // 2  # 16
    if delta < half:
        return delta
    else:
        log_ratio = np.log(delta / half) / np.log(max_distance / half)
        bucket = half + int(log_ratio * half)
        return min(bucket, num_buckets - 1)

# Compute bucket mapping for all offsets 0..255
max_offset = 256
offsets = np.arange(max_offset)
buckets = np.array([t5_bucket(d) for d in offsets])

print("=== T5 Relative Bias Bucketing ===")
print(f"num_buckets=32, max_distance=128\n")
print(f"{'Offset δ':>10} {'Bucket':>8} {'Resolution':>12}")
print("-" * 35)
prev_bucket = -1
for delta in [0, 1, 2, 5, 10, 15, 16, 20, 32, 48, 64, 96, 128, 192, 255]:
    b = t5_bucket(delta)
    res = "exact" if delta < 16 else "log"
    print(f"{delta:>10} {b:>8} {res:>12}")

# Count offsets per bucket
print(f"\n=== Offsets per Bucket ===")
for b in range(32):
    count = np.sum(buckets == b)
    range_offsets = np.where(buckets == b)[0]
    if len(range_offsets) > 0:
        span = f"[{range_offsets[0]}-{range_offsets[-1]}]"
    else:
        span = "empty"
    if count <= 10:
        print(f"  Bucket {b:>2}: {count:>4} offsets  {span}")
    elif b < 20 or b >= 30:
        print(f"  Bucket {b:>2}: {count:>4} offsets  {span}")

## 7. RoPE Scaling for Long Context

Three main approaches to extend RoPE beyond training length:
1. **Position Interpolation (PI)**: scale positions by s = n_train/n_test 
2. **NTK-Aware**: scale the base instead of positions
3. **YaRN**: non-uniform per-dimension scaling with correction factor

In [ ]:
# === 7a. Position Interpolation vs NTK-Aware Scaling ===

d = 64
base = 10000
n_train = 4096
n_test = 16384  # 4× extension
scale = n_test / n_train  # s = 4

dim_pairs = np.arange(d // 2)

# Original RoPE frequencies
theta_orig = 1.0 / (base ** (2 * dim_pairs / d))

# Position Interpolation: scale positions → effectively divide frequencies by s
theta_pi = theta_orig / scale

# NTK-Aware: scale the base
ntk_base = base * scale ** (d / (d - 2))
theta_ntk = 1.0 / (ntk_base ** (2 * dim_pairs / d))

print(f"=== RoPE Scaling Comparison ===")
print(f"d={d}, base={base}, n_train={n_train}, n_test={n_test}, scale={scale}×\n")
print(f"NTK scaled base: {ntk_base:.1f} (vs original {base})\n")

print(f"{'Pair i':>8} {'θ_orig':>12} {'θ_PI':>12} {'θ_NTK':>12} {'PI ratio':>10} {'NTK ratio':>10}")
print("-" * 68)
for i in [0, 4, 8, 16, 24, 31]:
    pi_ratio = theta_pi[i] / theta_orig[i]
    ntk_ratio = theta_ntk[i] / theta_orig[i]
    print(f"{i:>8} {theta_orig[i]:>12.6f} {theta_pi[i]:>12.6f} {theta_ntk[i]:>12.6f} {pi_ratio:>10.4f} {ntk_ratio:>10.4f}")

print(f"\nKey difference:")
print(f"  PI:  ALL frequencies divided by {scale} uniformly → compresses local resolution")
print(f"  NTK: High-freq dims scaled more, low-freq barely touched → preserves local info")

In [ ]:
# === 7b. YaRN — Non-Uniform Scaling ===

def yarn_frequencies(d, base, n_train, n_test, beta_fast=32, beta_slow=1):
    """
    Compute YaRN-scaled RoPE frequencies.
    
    Three zones:
    - High frequency (λ < n_train * beta_fast_ratio): interpolate
    - Mid frequency: smooth blend  
    - Low frequency (λ > n_train * beta_slow_ratio): no scaling
    """
    scale = n_test / n_train
    dim_pairs = np.arange(d // 2)
    theta_orig = 1.0 / (base ** (2 * dim_pairs / d))
    wavelengths = 2 * np.pi / theta_orig
    
    # Compute ramp function γ(i) ∈ [0, 1]
    # γ = 0 → full interpolation; γ = 1 → no scaling
    low = n_train / beta_fast   # wavelength threshold for high freq
    high = n_train / beta_slow  # wavelength threshold for low freq
    
    gamma = np.clip((wavelengths - low) / (high - low), 0, 1)
    
    # Scaled frequencies: blend between interpolation (θ/s) and no-scaling (θ)
    theta_yarn = theta_orig * (1.0 / ((1 - gamma) * scale + gamma))
    
    return theta_yarn, gamma

d = 64
base = 10000
n_train = 4096
n_test = 16384
scale = n_test / n_train

theta_orig = 1.0 / (base ** (2 * np.arange(d // 2) / d))
theta_yarn, gamma = yarn_frequencies(d, base, n_train, n_test)
theta_pi = theta_orig / scale
theta_ntk_base = base * scale ** (d / (d - 2))
theta_ntk = 1.0 / (theta_ntk_base ** (2 * np.arange(d // 2) / d))

wavelengths = 2 * np.pi / theta_orig

print(f"=== YaRN Scaling (d={d}, {n_train} → {n_test}, scale={scale}×) ===\n")
print(f"{'Pair i':>8} {'γ(i)':>8} {'θ_orig':>10} {'θ_yarn':>10} {'λ_orig':>10} {'Zone':>12}")
print("-" * 62)
for i in range(d // 2):
    zone = "INTERPOLATE" if gamma[i] < 0.1 else ("BLEND" if gamma[i] < 0.9 else "NO SCALE")
    if i % 4 == 0 or i >= d//2 - 2:
        print(f"{i:>8} {gamma[i]:>8.3f} {theta_orig[i]:>10.6f} {theta_yarn[i]:>10.6f} {wavelengths[i]:>10.0f} {zone:>12}")

# Verify: RoPE dot product still depends on relative position
np.random.seed(42)
q = np.random.randn(d)
k = np.random.randn(d)

def dot_with_freqs(q, k, m, n, freqs):
    """Compute dot(R_m q, R_n k) with given frequencies."""
    q_rot = np.zeros_like(q)
    k_rot = np.zeros_like(k)
    for i in range(len(freqs)):
        angle_m = m * freqs[i]
        angle_n = n * freqs[i]
        cm, sm = np.cos(angle_m), np.sin(angle_m)
        cn, sn = np.cos(angle_n), np.sin(angle_n)
        q_rot[2*i] = q[2*i]*cm - q[2*i+1]*sm
        q_rot[2*i+1] = q[2*i]*sm + q[2*i+1]*cm
        k_rot[2*i] = k[2*i]*cn - k[2*i+1]*sn
        k_rot[2*i+1] = k[2*i]*sn + k[2*i+1]*cn
    return q_rot @ k_rot

print(f"\n=== YaRN preserves relative position property ===")
for offset in [0, 1, 5, 10]:
    dots = [dot_with_freqs(q, k, m, m - offset, theta_yarn) for m in range(offset, offset + 5)]
    spread = max(dots) - min(dots)
    print(f"  offset={offset:>3}: spread = {spread:.2e} {'✓' if spread < 1e-10 else '✗'}")

In [ ]:
# === 7c. RoPE Scaling Visualisation ===

if HAS_MPL:
    d = 64
    base = 10000
    n_train = 4096
    dim_pairs = np.arange(d // 2)
    theta_orig = 1.0 / (base ** (2 * dim_pairs / d))
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    for scale_factor, ax in zip([2, 4, 8], axes):
        n_test = n_train * scale_factor
        
        # Compute all scaling variants
        theta_pi = theta_orig / scale_factor
        ntk_b = base * scale_factor ** (d / (d - 2))
        theta_ntk = 1.0 / (ntk_b ** (2 * dim_pairs / d))
        theta_yarn, _ = yarn_frequencies(d, base, n_train, n_test)
        
        ax.semilogy(dim_pairs, theta_orig, 'k-', linewidth=2, label='Original', alpha=0.7)
        ax.semilogy(dim_pairs, theta_pi, 'r--', linewidth=1.5, label=f'PI (÷{scale_factor})', alpha=0.7)
        ax.semilogy(dim_pairs, theta_ntk, 'g-.', linewidth=1.5, label='NTK', alpha=0.7)
        ax.semilogy(dim_pairs, theta_yarn, 'b:', linewidth=2, label='YaRN', alpha=0.9)
        
        ax.set_xlabel('Dimension pair i')
        ax.set_ylabel('Frequency θ_i')
        ax.set_title(f'{scale_factor}× extension ({n_train}→{n_test})')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('RoPE Scaling Methods: Frequency Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available")

## 8. Extrapolation Comparison

Compare how sinusoidal PE, RoPE (unscaled), and ALiBi behave when sequence length exceeds training length. We measure the dot-product pattern shift for each method.

In [ ]:
# === 8. Extrapolation Comparison ===

d = 64
base = 10000
n_train = 128
n_test = 512  # 4× extrapolation

np.random.seed(42)
q = np.random.randn(d) * 0.3
k = np.random.randn(d) * 0.3

# --- Sinusoidal PE: dot product of position vectors ---
pe_long = sinusoidal_pe(n_test, d, base)
sin_dots = np.array([pe_long[0] @ pe_long[pos] for pos in range(n_test)])

# --- RoPE: dot product of rotated q,k ---
thetas = compute_rope_freqs(d, base)
rope_dots = []
for pos in range(n_test):
    R0 = build_rope_matrix(0, thetas)
    Rp = build_rope_matrix(pos, thetas)
    dot_val = (R0 @ q) @ (Rp @ k)
    rope_dots.append(dot_val)
rope_dots = np.array(rope_dots)

# --- ALiBi: fixed content score + linear penalty ---
slopes = alibi_slopes(8)
content_score = q @ k  # fixed content score
alibi_scores = {}
for h_idx, h in enumerate([0, 3, 7]):
    alibi_scores[f'h{h+1}'] = content_score - slopes[h] * np.arange(n_test)

print(f"=== Extrapolation Comparison ===")
print(f"n_train={n_train}, n_test={n_test} ({n_test/n_train:.0f}× extrapolation)\n")

print(f"{'Position':>10} {'Sinusoidal':>14} {'RoPE':>14} {'ALiBi h1':>14} {'ALiBi h8':>14}")
print("-" * 70)
for pos in [0, 10, 50, 100, 127, 128, 200, 300, 400, 511]:
    in_train = "✓" if pos < n_train else "✗"
    print(f"{pos:>10} {sin_dots[pos]:>14.4f} {rope_dots[pos]:>14.4f} "
          f"{alibi_scores['h1'][pos]:>14.4f} {alibi_scores['h8'][pos]:>14.4f}  {in_train}")

print(f"\n=== Key Observations ===")
print(f"  Sinusoidal: dot product oscillates but stays bounded → no catastrophic failure,")
print(f"              but model may not have learned to interpret these PE vectors")
print(f"  RoPE:       dot product pattern continues smoothly → position info preserved,")
print(f"              but high-freq dimensions may alias beyond n_train")
print(f"  ALiBi:      linear penalty simply grows → smooth extrapolation,")
print(f"              no new representations, just larger penalties")

In [ ]:
# === 8b. Extrapolation Visualisation ===

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    positions = np.arange(n_test)
    
    # (a) Sinusoidal dot product
    axes[0].plot(positions, sin_dots, 'b-', alpha=0.7, linewidth=0.5)
    axes[0].axvline(x=n_train, color='red', ls='--', alpha=0.7, label=f'n_train={n_train}')
    axes[0].set_xlabel('Position')
    axes[0].set_ylabel('PE(0) · PE(pos)')
    axes[0].set_title('Sinusoidal PE')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # (b) RoPE dot product
    axes[1].plot(positions, rope_dots, 'g-', alpha=0.7, linewidth=0.5)
    axes[1].axvline(x=n_train, color='red', ls='--', alpha=0.7, label=f'n_train={n_train}')
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('dot(R₀q, Rₚk)')
    axes[1].set_title('RoPE (unscaled)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # (c) ALiBi scores for different heads
    colors = ['red', 'blue', 'green']
    for (name, scores), color in zip(alibi_scores.items(), colors):
        axes[2].plot(positions, scores, color=color, alpha=0.7, label=name, linewidth=1.5)
    axes[2].axvline(x=n_train, color='red', ls='--', alpha=0.7, label=f'n_train={n_train}')
    axes[2].set_xlabel('Position')
    axes[2].set_ylabel('Content score − m·distance')
    axes[2].set_title('ALiBi')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'Extrapolation Behavior (n_train={n_train}, n_test={n_test})', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available")

## 9. RoPE in Full Attention & Integration with Scaled Dot-Product

Putting it all together: RoPE applies rotation to Q and K *before* computing attention scores.
Compare attention patterns with no PE, sinusoidal PE, RoPE, and ALiBi.

In [ ]:
# === 9. Full Attention with Different PE Methods ===

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

def attention_no_pe(X, W_Q, W_K, W_V):
    """Standard attention with no positional encoding."""
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    # Causal mask
    mask = np.triu(np.ones_like(scores) * (-1e9), k=1)
    weights = softmax(scores + mask)
    return weights, weights @ V

def attention_sinusoidal(X, W_Q, W_K, W_V, pe):
    """Attention with sinusoidal PE added to input."""
    X_pe = X + pe[:X.shape[0]]
    return attention_no_pe(X_pe, W_Q, W_K, W_V)

def attention_rope(X, W_Q, W_K, W_V, base=10000):
    """Attention with RoPE applied to Q and K."""
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    n = Q.shape[0]
    positions = np.arange(n, dtype=float)
    Q_rot = apply_rope_efficient(Q, positions, base)
    K_rot = apply_rope_efficient(K, positions, base)
    d_k = Q.shape[-1]
    scores = Q_rot @ K_rot.T / np.sqrt(d_k)
    mask = np.triu(np.ones_like(scores) * (-1e9), k=1)
    weights = softmax(scores + mask)
    return weights, weights @ V

def attention_alibi(X, W_Q, W_K, W_V, slope):
    """Attention with ALiBi bias."""
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    n = Q.shape[0]
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    # ALiBi bias
    dist = np.arange(n)[None, :] - np.arange(n)[:, None]
    dist = np.minimum(dist, 0)
    bias = dist * slope
    # Causal mask
    mask = np.triu(np.ones_like(scores) * (-1e9), k=1)
    weights = softmax(scores + bias + mask)
    return weights, weights @ V

# Setup
np.random.seed(42)
n, d, d_k = 12, 16, 8
X = np.random.randn(n, d) * 0.3
W_Q = np.random.randn(d, d_k) * 0.2
W_K = np.random.randn(d, d_k) * 0.2
W_V = np.random.randn(d, d_k) * 0.2
pe = sinusoidal_pe(n, d)

# Compute attention with each method
w_none, _ = attention_no_pe(X, W_Q, W_K, W_V)
w_sin, _  = attention_sinusoidal(X, W_Q, W_K, W_V, pe)
w_rope, _ = attention_rope(X, W_Q, W_K, W_V)
w_ali_local, _ = attention_alibi(X, W_Q, W_K, W_V, slope=0.5)
w_ali_global, _ = attention_alibi(X, W_Q, W_K, W_V, slope=0.004)

print("=== Attention Weights from Last Token (pos 11) ===")
print(f"{'Key pos':>8} {'No PE':>10} {'Sinusoidal':>12} {'RoPE':>10} {'ALiBi loc':>11} {'ALiBi glob':>12}")
print("-" * 68)
for j in range(n):
    print(f"{j:>8} {w_none[-1,j]:>10.4f} {w_sin[-1,j]:>12.4f} {w_rope[-1,j]:>10.4f} "
          f"{w_ali_local[-1,j]:>11.4f} {w_ali_global[-1,j]:>12.4f}")

# Entropy comparison
def entropy(w):
    w = w + 1e-12
    return -np.sum(w * np.log2(w), axis=-1)

print(f"\n=== Attention Entropy (last token) ===")
for name, w in [("No PE", w_none), ("Sinusoidal", w_sin), ("RoPE", w_rope),
                ("ALiBi local", w_ali_local), ("ALiBi global", w_ali_global)]:
    h = entropy(w[-1])
    print(f"  {name:>14}: H = {h:.4f} bits (max = {np.log2(n):.4f})")

In [ ]:
# === 9b. Attention Pattern Visualisation ===

if HAS_MPL:
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    titles = ['No PE', 'Sinusoidal PE', 'RoPE', 'ALiBi (local)', 'ALiBi (global)']
    matrices = [w_none, w_sin, w_rope, w_ali_local, w_ali_global]
    
    for ax, title, w in zip(axes, titles, matrices):
        im = ax.imshow(w, aspect='auto', cmap='Blues', vmin=0)
        ax.set_xlabel('Key position')
        ax.set_ylabel('Query position')
        ax.set_title(title, fontsize=10)
    
    plt.colorbar(im, ax=axes[-1], shrink=0.8, label='Attention weight')
    plt.suptitle('Attention Patterns with Different Positional Encodings', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available")

## 10. Permutation Invariance Proof

Formal demonstration that attention without PE is permutation-invariant:
if X' = PX for permutation P, then scores S' = PSP^T.

In [ ]:
# === 10. Permutation Invariance Proof ===

np.random.seed(42)
n, d, d_k = 6, 8, 4

X = np.random.randn(n, d)
W_Q = np.random.randn(d, d_k) * 0.3
W_K = np.random.randn(d, d_k) * 0.3

# Original scores
Q = X @ W_Q
K = X @ W_K
S = Q @ K.T / np.sqrt(d_k)

# Random permutation
perm = np.array([3, 0, 5, 1, 4, 2])
P = np.zeros((n, n))
for i, j in enumerate(perm):
    P[i, j] = 1.0

# Permuted input
X_perm = P @ X
Q_perm = X_perm @ W_Q
K_perm = X_perm @ W_K
S_perm = Q_perm @ K_perm.T / np.sqrt(d_k)

# Verify S' = P S P^T
S_theory = P @ S @ P.T
error = np.linalg.norm(S_perm - S_theory)

print("=== Permutation Invariance Proof ===")
print(f"Permutation: {list(perm)}")
print(f"\n||S' - PSP^T|| = {error:.2e}")
print(f"Scores are just reordered, not changed! ✓")

# Attention weights (no mask, no positional info)
W_orig = softmax(S)
W_perm = softmax(S_perm)
W_theory = P @ W_orig @ P.T
print(f"\n||W' - PWP^T|| = {np.linalg.norm(W_perm - W_theory):.2e}")
print(f"Attention weights are also just reordered ✓")

# Now show PE breaks this
pe = sinusoidal_pe(n, d)
X_pe = X + pe
X_perm_pe = P @ X + pe  # Note: PE NOT permuted

Q_pe = X_pe @ W_Q
K_pe = X_pe @ W_K
S_pe = Q_pe @ K_pe.T / np.sqrt(d_k)

Q_perm_pe = X_perm_pe @ W_Q
K_perm_pe = X_perm_pe @ W_K
S_perm_pe = Q_perm_pe @ K_perm_pe.T / np.sqrt(d_k)

S_would_be = P @ S_pe @ P.T
error_with_pe = np.linalg.norm(S_perm_pe - S_would_be)
print(f"\nWith PE: ||S' - PSP^T|| = {error_with_pe:.4f}")
print(f"PE BREAKS permutation invariance → positional information preserved ✓")

## 11. PE Comparison Dashboard

Summary comparison of all PE methods: sinusoidal, learned, RoPE, ALiBi, T5 bias — parameters, extrapolation, computation, and adoption.

In [ ]:
# === 11. PE Comparison Dashboard ===

print("=" * 95)
print("POSITIONAL ENCODING COMPARISON")
print("=" * 95)

headers = ["Method", "Type", "Injection", "Extra Params", "Extrapolation", "Used By (2025)"]
rows = [
    ["Sinusoidal",   "Absolute", "Additive (embed)", "0",         "Poor",      "Original Transformer"],
    ["Learned Abs",  "Absolute", "Additive (embed)", "L × d",     "None",      "BERT, GPT-2"],
    ["Shaw Rel",     "Relative", "Additive (score)", "2(2k+1)dₖ", "Moderate",  "Research"],
    ["T-XL",         "Relative", "Additive (score)", "2dₖ + bias", "Good",     "Transformer-XL"],
    ["T5 Bias",      "Relative", "Additive (score)", "32 × H",    "Good",      "T5, Flan-T5"],
    ["ALiBi",        "Relative", "Bias (score)",     "0",         "Strong",    "BLOOM, MPT"],
    ["RoPE",         "Relative", "Rotation (Q,K)",   "0",         "Moderate*", "LLaMA, Mistral, Gemma"],
    ["RoPE+YaRN",    "Relative", "Rotation (Q,K)",   "0",         "Strong",    "LLaMA-3 long ctx"],
    ["xPos",         "Relative", "Rotation+decay",   "0",         "Strong",    "Research"],
    ["CoPE",         "Context",  "Context-gated",    "d",         "Unknown",   "Research"],
    ["NoPE",         "—",        "Causal mask only",  "0",         "Poor",      "Research"],
]

# Print formatted table
widths = [12, 10, 18, 12, 14, 24]
header_str = " | ".join(f"{h:<{w}}" for h, w in zip(headers, widths))
print(header_str)
print("-" * len(header_str))
for row in rows:
    print(" | ".join(f"{v:<{w}}" for v, w in zip(row, widths)))

print("\n* RoPE extrapolation is 'Moderate' without scaling; 'Strong' with YaRN/NTK")

# Parameter counts for real models
print(f"\n{'=' * 60}")
print("PE PARAMETER COUNTS FOR REAL MODELS")
print(f"{'=' * 60}")
models = [
    ("BERT-base",      512,  768,  "Learned", 512 * 768),
    ("GPT-2",          1024, 768,  "Learned", 1024 * 768),
    ("GPT-3 (175B)",   2048, 12288,"Learned", 2048 * 12288),
    ("LLaMA-3 8B",     8192, 4096, "RoPE",    0),
    ("LLaMA-3 70B",    8192, 8192, "RoPE",    0),
    ("Mistral 7B",     32768,4096, "RoPE",    0),
    ("BLOOM 176B",     2048, 14336,"ALiBi",   0),
    ("T5-base",        512,  768,  "T5 Bias", 32 * 12),
]

print(f"{'Model':<18} {'Max L':>7} {'d_model':>8} {'PE Type':<10} {'PE Params':>12} {'% of 1B':>8}")
print("-" * 68)
for name, L, d, pe_type, params in models:
    pct = params / 1e9 * 100
    print(f"{name:<18} {L:>7} {d:>8} {pe_type:<10} {params:>12,} {pct:>7.4f}%")

## Summary

| Topic | Key Result |
|---|---|
| Permutation invariance | Without PE, attention produces identical scores for any token permutation |
| Sinusoidal PE | Multi-frequency sin/cos; PE(pos+k) = M_k·PE(pos); dot product depends on offset |
| Learned PE | Simple but hard ceiling at L; undertrained tail problem |
| RoPE | Block-diagonal rotation; dot(R_m q, R_n k) = f(m−n); dominant standard |
| RoPE frequencies | d/2 wavelengths from 2π to 2π·base; slow dims barely rotate → dead for short ctx |
| ALiBi | Linear penalty −m_h·(i−j); zero parameters; strong extrapolation |
| T5 bias | Logarithmic bucketing; exact nearby, coarse distant |
| PI / NTK / YaRN | Scale positions, base, or per-dimension; YaRN best quality/cost tradeoff |
| Extrapolation | ALiBi smoothest; RoPE+YaRN strong; sinusoidal/learned fail |

**Next**: [exercises.ipynb](exercises.ipynb) — 8 hands-on exercises covering PE by hand, RoPE, ALiBi, frequency analysis, and YaRN zones.